In [1]:
import phonlp
from src.triplet_extraction.src import init_vncorenlp, load_synonym_dict, load_stopwords
import os

current_dir = os.getcwd()
base_dir = os.path.dirname(current_dir)
print(f"Working directory: {current_dir}")
print(f"Base directory set to: {base_dir}\n")

# === Define files paths relative to base directory ===
vncorenlp_dir = os.path.join(current_dir, "nlp_models", "VnCoreNLP-1.2")
phonlp_dir = os.path.join(current_dir, "nlp_models", "phonlp")
synonym_file = os.path.join(current_dir, "listSameKey.txt")
stopwords_file = os.path.join(current_dir, "stopwords.csv")
no_triplet_csv_path = os.path.join(current_dir, "logs", "no_triplets_dat_dai_log_1.csv")
log_file_path = os.path.join(current_dir, "logs", "dat_dai_triplet_extraction.txt")

# === Initialize NLP models ===
vncorenlp_client = init_vncorenlp(vncorenlp_dir)
phoNLP_model = phonlp.load(save_dir=phonlp_dir)
synonym_dict = load_synonym_dict(synonym_file)
stopwords = load_stopwords(stopwords_file)

Working directory: E:\Github\LawAssistant\triplet_extraction
Base directory set to: E:\Github\LawAssistant

Loading model from: E:\Github\LawAssistant\triplet_extraction\nlp_models\phonlp/phonlp.pt


In [2]:
from src.triplet_extraction.src import init_mongo
# === Initialize MongoDB ===
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
documents_col = db["documents"]
sections_col = db["legal_sections"]
process_sections_col = db["processed_legal_sections"]
relations_col = db["relations"]

You successfully connected to MongoDB!


In [7]:


sentence = "Phải điều chỉnh thời hạn sử dụng đất."

os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
logger, console_handler, file_handler = setup_logger(
    name="triplet_extraction",
    level=logging.DEBUG,
    log_to_file=True,
    file_path=log_file_path
)

triplets = triplet_extraction(
    text=sentence,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    stopwords=stopwords,
    logger=logger,
    max_depth=3,
)

for t in triplets:
    print(t)

[INFO] Logging to files: E:\Github\LawAssistant\triplet_extraction\logs\dat_dai_triplet_extraction.txt
100%|██████████| 1/1 [00:00<00:00, 14.06it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] [{'id': 1, 'word': 'phải', 'pos': 'V', 'head': 0, 'deprel': 'root'}]
[DEBUG] -----------------VP-----------------
[DEBUG] [{'id': 2, 'word': 'điều_chỉnh', 'pos': 'V', 'head': 1, 'deprel': 'vmod'}, {'id': 3, 'word': 'thời_hạn', 'pos': 'N', 'head': 2, 'deprel': 'dob'}, {'id': 4, 'word': 'sử_dụng', 'pos': 'V', 'head': 3, 'deprel': 'nmod'}, {'id': 5, 'word': 'đất', 'pos': 'N', 'head': 4, 'deprel': 'dob'}, {'id': 6, 'word': '.', 'pos': 'CH', 'head': 1, 'deprel': 'punct'}]
[DEBUG] -----------------subjects----------------
[DEBUG] ['phải']
[DEBUG] -----------------verbs----------------
[DEBUG] điều_chỉnh
[DEBUG] -----------------objects----------------
[DEBUG] thời_hạn sử_dụng đất


 id       word pos head deprel
  1       phải   V    0   root
  2 điều_chỉnh   V    1   vmod
  3   thời_hạn   N    2    dob
  4    sử_dụng   V    3   nmod
  5        đất   N    4    dob
  6          .  CH    1  punct


100%|██████████| 1/1 [00:00<00:00, 17.86it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] []
[DEBUG] -----------------VP-----------------
[DEBUG] []
[DEBUG] -----------------subjects----------------
[DEBUG] []
[DEBUG] -----------------verbs----------------
[DEBUG] -----------------objects----------------
100%|██████████| 1/1 [00:00<00:00, 18.81it/s]
[DEBUG] -----------------NP-----------------
[DEBUG] [{'id': 1, 'word': 'thời_hạn', 'pos': 'N', 'head': 0, 'deprel': 'root'}]
[DEBUG] -----------------VP-----------------
[DEBUG] [{'id': 2, 'word': 'sử_dụng', 'pos': 'V', 'head': 1, 'deprel': 'nmod'}, {'id': 3, 'word': 'đất', 'pos': 'N', 'head': 1, 'deprel': 'nmod'}]
[DEBUG] -----------------subjects----------------
[DEBUG] ['thời_hạn']
[DEBUG] -----------------verbs----------------
[DEBUG] sử_dụng
[DEBUG] -----------------objects----------------
[DEBUG] đất


('phải', 'điều chỉnh', 'thời hạn')
('thời hạn', 'sử dụng', 'đất')


In [ ]:
sections = sections_col.find(
    {
        "is_amendment": True,
        "type": {"$in": ["điểm"]},
    }
)
for sec in sections:
    print(sec['_id'])

In [6]:
import re
import csv
from collections import defaultdict
from bson import ObjectId

logs_file = r"/src/triplet_extraction/logs/dat_dai_triplet_extraction_08_01_2026.txt"
output_file = r"/src/triplet_extraction/logs/log_not_in_triplet_with_info.csv"

# ------------------------
# 1. Parse log file
# ------------------------
log_count = defaultdict(int)
log_pattern = re.compile(r"Processing section_id:\s*([0-9a-f]{24})")

with open(logs_file, "r", encoding="utf-8") as f:
    for line in f:
        m = log_pattern.search(line)
        if m:
            log_count[m.group(1)] += 1

log_ids = set(log_count.keys())
print(f"Unique section_ids in log: {len(log_ids)}")

# ------------------------
# 2. Get relation / triplet document_ids
# ------------------------
relation_ids = set(
    relations_col.distinct("documents.document_id")
)
print(f"Unique section_ids in triplets: {len(relation_ids)}")

# ------------------------
# 3. Compare (LOG but NOT in TRIPLET)
# ------------------------
missing_in_triplet = log_ids - relation_ids
print(f"❌ In log but NOT in triplets: {len(missing_in_triplet)}")

# ------------------------
# 4. Query section info & save to CSV
# ------------------------
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "process_id",
        "section_id",
        "so_hieu",
        "sequence",
        "content"
    ])

    cursor = process_sections_col.find(
        {"section_id": {"$in": [ObjectId(x) for x in missing_in_triplet]}},
        {
            "section_id": 1,
            "so_hieu": 1,
            "sequence": 1,
            "content": 1
        }
    )

    for s in cursor:
        writer.writerow([
            str(s.get("_id")),
            s.get("section_id", ""),
            s.get("so_hieu", ""),
            s.get("sequence", ""),
            s.get("content", "")
        ])

print("✅ DONE")
print(f"Detailed log-only sections written to:\n{output_file}")

Unique section_ids in log: 8574
Unique section_ids in triplets: 8261
❌ In log but NOT in triplets: 313
✅ DONE
Detailed log-only sections written to:
E:\Github\LawAssistant\triplet_extraction\logs\log_not_in_triplet_with_info.csv


In [ ]:
from bson import ObjectId

SECTION_ID = "694fb81aaedc69db48c74462"

sections = process_sections_col.find(
    {"section_id": ObjectId(SECTION_ID)}
)
sections = list(sections)
for sec in sections:
    print(sec)

In [ ]:
import re
import csv
from collections import defaultdict

logs_file = r"/src/triplet_extraction/logs/dat_dai_triplet_extraction_08_01_2026.txt"
no_triplet_file = r"/src/triplet_extraction/logs/no_triplets_dat_dai_log_2.csv"

log_count = defaultdict(int)
log_pattern = re.compile(
    r"Processing section_id:\s*([0-9a-f]{24})"
)

with open(logs_file, "r", encoding="utf-8") as f:
    for line in f:
        m = log_pattern.search(line)
        if m:
            log_count[m.group(1)] += 1

print(f"Unique section_ids in log: {len(log_count)}")


ids = triplets_col.distinct("documents.document_id")
ids = set(ids)
print(f"Unique section_ids in triplets: {len(ids)}")


with open(no_triplet_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["section_id", "document_number", "sequence", "sentence"])

    for r in redo_rows:
        writer.writerow([
            r["section_id"],
            r["document_number"],
            r["sequence"],
            r["sentence"]
        ])

print("✅ DONE")
print(f"Redo list written to:\n{no_triplet_file}")

In [3]:
from src.triplet_extraction.src import clean_text, parsing_result


def parse_dataframe_to_tokens(df):
    """Convert DataFrame to a list of token dicts"""
    tokens = []
    for _, row in df.iterrows():
        token = {
            'id': int(row['id']),
            'word': str(row['word']),
            'pos': str(row['pos']),
            'head': int(row['head']),
            'deprel': str(row['deprel'])
        }
        tokens.append(token)
    return tokens


def split_sentence_np_vp(tokens):
    if not tokens:
        return [], []

    root_index = -1
    root_id = None

    # Find the main verb (root or first valid verb)
    for i, token in enumerate(tokens):
        if token['pos'] == 'V':
            if token['deprel'] == 'root' and token['head'] == 0:
                # Avoid picking verb at start (index 0)
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']
                break
            elif root_index == -1 and token['deprel'] != 'nmod':
                # Avoid first word if it's a verb
                if i == 0:
                    continue
                root_index = i
                root_id = token['id']

    # Fallback – pick next verb if root not found
    if root_index == -1:
        for i, token in enumerate(tokens):
            if token['pos'] == 'V' and i > 0:  # skip first position
                root_index = i
                root_id = token['id']
                break

    # Final split
    if root_index != -1 and root_id is not None:
        np_tokens = tokens[:root_index]
        vp_tokens = tokens[root_index:]
        return np_tokens, vp_tokens

    return [], []

def collect_dependents(tokens, head_id):
    """Return set of token ids: head_id + all recursive dependents"""
    subtree = {head_id}
    result = []
    added = True
    while added:
        added = False
        for token in tokens:
            if token['head'] in subtree and token['id'] not in subtree:
                subtree.add(token['id'])
                result.append(token)
                added = True
    return result


def collect_direct_dependents(tokens, head_id):
    """Return list of token dicts that directly depend on head_id"""
    return [t for t in tokens if t['head'] == head_id]


def rebuild_phrase(tokens):
    """Sort tokens by their original position in the sentence and join them together"""
    tokens_sorted = sorted(tokens, key=lambda x: x['id'])
    phrase = " ".join(t['word'] for t in tokens_sorted)
    return phrase

def extract_main_subjects(np_tokens):
    if not np_tokens:
        return []

    sub_tokens = [t for t in np_tokens if t['deprel'] == 'sub']
    if not sub_tokens:
        sub_tokens = [t for t in np_tokens if t['deprel'] == 'root']
    if not sub_tokens:
        return []

    main_subjects = [sub_tokens[0]]
    main_subjects.extend(collect_direct_dependents(np_tokens, sub_tokens[0]['id']))
    if main_subjects and main_subjects[-1]['pos'] in ['Cc', 'CH']:
        main_subjects.pop()
    if len(main_subjects) == len(np_tokens):
        return [rebuild_phrase(np_tokens)]

    # Find Coordination Word (Cc, CH)
    coord_tokens = [t for t in np_tokens if (t['pos'] in ['Cc', 'CH'] and t not in main_subjects)]
    non_main_tokens = set()
    if len(coord_tokens) > 0:
        phrases = []

        for coord in coord_tokens:
            coord_index = next((i for i, t in enumerate(np_tokens) if t['id'] == coord['id']), None)

            left_tokens = []
            main_subjects_id = [obj['id'] for obj in main_subjects]
            for i in range(coord_index - 1, -1, -1):
                token = np_tokens[i]
                if token['pos'] not in ['CH', 'Cc'] and token['id'] not in main_subjects_id:
                    left_tokens.append(token)
                    non_main_tokens.add(token['id'])
                else:
                    break

            right_tokens = []
            for i in range(coord_index + 1, len(np_tokens)):
                token = np_tokens[i]
                if token['pos'] not in ['CH', 'Cc']:
                    right_tokens.append(token)
                    non_main_tokens.add(token['id'])
                else:
                    break

            if left_tokens:
                phrases.append(rebuild_phrase(left_tokens))
            if right_tokens:
                phrases.append(rebuild_phrase(right_tokens))

        # Remove duplicates while preserving order
        phrases = list(dict.fromkeys(phrases))

        for sub in main_subjects:
            if sub['id'] in non_main_tokens:
                main_subjects.remove(sub)
        main_subject_phrase = rebuild_phrase(main_subjects)

        # Properly combine main subject with each phrase
        combined_phrases = []
        for phrase in phrases:
            combined_phrases.append(main_subject_phrase + " " + phrase)

        if not combined_phrases:
            return [main_subject_phrase]

        return combined_phrases
    else:
        return [rebuild_phrase(np_tokens)]

def extract_verbs(vp_tokens):
    if not vp_tokens:
        return [], []

    # Find the root verb first
    root_verb = None
    for t in vp_tokens:
        if t['deprel'] == 'root' and t['head'] == 0 and t['pos'] == 'V':
            root_verb = t
            break

    # Fallback to the first verb that not nmod or aux
    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V' and t['deprel'] not in ['nmod', 'aux']:
                root_verb = t
                break

    # Fallback to any first verb in vp_tokens
    if not root_verb:
        for t in vp_tokens:
            if t['pos'] == 'V':
                root_verb = t
                break

    if not root_verb:
        return [vp_tokens[0]['word']], [vp_tokens[0]]

    # Check for coordination markers (CH, Cc) that are direct dependents of root
    coord_markers = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH'] and t['head'] == root_verb['id']]

    if coord_markers:
        coordinated_verbs = [root_verb]
        for t in vp_tokens:
            if t['pos'] == 'V' and t['head'] == root_verb['id'] and t['deprel'] in ['vmod', 'conj']:
                coordinated_verbs.append(t)

        # Sort by ID to maintain order
        coordinated_verbs.sort(key=lambda x: x['id'])

        verb_phrases = []
        all_tokens = []
        coordinated_verbs_id = [v['id'] for v in coordinated_verbs]
        for verb in coordinated_verbs:
            phrase_tokens = [verb]
            dependents = collect_direct_dependents(vp_tokens, verb['id'])

            # Keep only dependents that are not other coordinated verbs or coordination markers
            dependents = [d for d in dependents if
                          d['id'] not in coordinated_verbs_id
                          and d['pos'] not in ['CH', 'Cc']
                          and d['deprel'] == 'vmod']

            phrase_tokens.extend(dependents)
            all_tokens.extend(phrase_tokens)

            verb_phrases.append({
                'text': rebuild_phrase(phrase_tokens),
                'tokens': phrase_tokens
            })

        return verb_phrases, all_tokens

    # Single verb: return it with its dependents
    verb_tokens = [root_verb]
    verb_tokens.extend(collect_direct_dependents(vp_tokens, root_verb['id']))
    sorted_verb_tokens = sorted(verb_tokens, key=lambda x: x['id'])

    # Filter out tokens after the first noun
    filtered_tokens = []
    for token in sorted_verb_tokens:
        if token['pos'].startswith('N'):
            break
        filtered_tokens.append(token)

    # If we filtered out everything, at least return the root verb
    if not filtered_tokens:
        filtered_tokens = [root_verb]

    return [{
        'text': rebuild_phrase(filtered_tokens),
        'tokens': filtered_tokens
    }], filtered_tokens

def extract_objects(vp_tokens, verb_token):
    # Remove verb tokens from vp_tokens (make a copy to avoid modifying during iteration)
    vp_tokens = [t for t in vp_tokens if t['id'] not in [v['id'] for v in verb_token]]

    if not vp_tokens:
        return []

    # Find the first object token (dob, iob, pob)
    obj_token = next((t for t in vp_tokens if t['deprel'] in ['dob', 'iob', 'pob']), None)

    # Fallback to the first noun in vp_tokens
    if obj_token is None:
        obj_token = next((t for t in vp_tokens if t['pos'] == 'N'), None)

    if obj_token is None:
        obj_token = next((t for t in vp_tokens if t['deprel'] == 'vmod'), None)

    if obj_token is None:
        return []

    # Collect main object and its dependents
    main_objects = [obj_token]
    main_objects.extend(collect_direct_dependents(vp_tokens, obj_token['id']))

    if len(main_objects) == len(vp_tokens):
        return [{
            'text': rebuild_phrase(vp_tokens),
            'tokens': vp_tokens
        }]

    # Find coordination tokens
    coord_tokens = [t for t in vp_tokens if t['pos'] in ['Cc', 'CH']]
    for obj in main_objects:
        if obj in coord_tokens:
            main_objects = []
            break

    if coord_tokens:
        combined_phrases = []

        for coord in coord_tokens:
            coord_index = next((i for i, t in enumerate(vp_tokens) if t['id'] == coord['id']), None)

            # LEFT TOKENS
            left_tokens = []
            for i in range(coord_index - 1, -1, -1):
                token = vp_tokens[i]
                if token['pos'] not in ['CH', 'Cc'] and token['id'] not in [obj['id'] for obj in main_objects]:
                    left_tokens.append(token)
                else:
                    break
            left_tokens = left_tokens[::-1]

            # RIGHT TOKENS
            right_tokens = []
            for i in range(coord_index + 1, len(vp_tokens)):
                token = vp_tokens[i]
                if token['pos'] not in ['CH', 'Cc']:
                    right_tokens.append(token)
                else:
                    break

            # Combine main object with left and right tokens
            for tokens_side in [left_tokens, right_tokens]:
                if tokens_side:
                    combined_phrases.append({
                        'text': rebuild_phrase(main_objects) + " " + rebuild_phrase(tokens_side),
                        'tokens': main_objects + tokens_side
                    })

        # Remove duplicates while preserving order
        seen = set()
        final_phrases = []
        for item in combined_phrases:
            if item['text'] not in seen:
                final_phrases.append(item)
                seen.add(item['text'])

        return final_phrases

    else:
        return [{
            'text': rebuild_phrase(vp_tokens),
            'tokens': vp_tokens
        }]


def process_sentence(df, logger):
    tokens = parse_dataframe_to_tokens(df)
    np_tokens, vp_tokens = split_sentence_np_vp(tokens)
    logger.debug("-----------------NP-----------------")
    logger.debug(np_tokens)
    logger.debug("-----------------VP-----------------")
    logger.debug(vp_tokens)

    # Extract subjects, verbs, objects
    subjects = extract_main_subjects(np_tokens)
    verbs, verbs_token = extract_verbs(vp_tokens)
    objects = extract_objects(vp_tokens, verbs_token)

    logger.debug("-----------------subjects----------------")
    logger.debug(subjects)
    logger.debug("-----------------verbs----------------")
    for verb in verbs:
        logger.debug(verb['text'])
    logger.debug("-----------------objects----------------")
    for obj in objects:
        logger.debug(obj['text'])

    verbs_position = {}
    for verb in verbs:
        verb_last_id = verb['tokens'][0]['id']
        verbs_position[verb['text']] = verb_last_id

    verbs_sorted = sorted(verbs, key=lambda v: v['tokens'][0]['id'])
    objects_sorted = sorted(objects, key=lambda o: o['tokens'][0]['id'])

    # Start combine them into triplets
    triplets = []
    for subj in subjects:
        for i, verb in enumerate(verbs_sorted):
            verb_last_id = verb['tokens'][-1]['id']

            # Determine the next verb's first ID (or infinity if this is the last verb)
            next_verb_first_id = verbs_sorted[i + 1]['tokens'][0]['id'] if i + 1 < len(verbs_sorted) else float('inf')

            # Objects that come after this verb but before the next verb
            obj_candidates = []
            for obj in objects_sorted:
                obj_id = obj['tokens'][-1]['id']
                if verb_last_id < obj_id < next_verb_first_id:
                    obj_candidates.append(obj)

            for obj in obj_candidates:
                triplets.append((subj, verb['text'], obj['text']))

    return triplets

def triplet_extraction(text, vncorenlp_client, phoNLP_model, stopwords, logger, max_depth=2, depth=0):
    """Recursively extract triplets from text, including nested subjects/objects"""
    if depth > max_depth or not text.strip():
        return []

    sentence = clean_text(text)
    segmented_text = vncorenlp_client.word_segment(sentence)

    # Annotate text
    annotation = phoNLP_model.annotate(text=segmented_text[0])
    df = parsing_result(annotation)
    print(df.to_string(index=False))

    triplets = process_sentence(df, logger)
    all_triplets = []

    for subj, verb, obj in triplets:
        # Refine subject
        try:
            subj_annotation = phoNLP_model.annotate(text=subj)
            df_subj = parsing_result(subj_annotation)
            refined_subj_triplets = process_sentence(df_subj, logger)
            if refined_subj_triplets and len(refined_subj_triplets) > 0 and len(refined_subj_triplets[0]) > 0:
                subj_refined = refined_subj_triplets[0][0]
            else:
                subj_refined = subj
        except (IndexError, Exception):
            subj_refined = subj
            refined_subj_triplets = []

        # Refine object
        try:
            obj_annotation = phoNLP_model.annotate(text=obj)
            df_obj = parsing_result(obj_annotation)
            refined_obj_triplets = process_sentence(df_obj, logger)
            if refined_obj_triplets and len(refined_obj_triplets) > 0 and len(refined_obj_triplets[0]) > 0:
                obj_refined = refined_obj_triplets[0][0]
            else:
                obj_refined = obj
        except (IndexError, Exception):
            obj_refined = obj
            refined_obj_triplets = []

        all_triplets.append((subj_refined, verb, obj_refined))
        all_triplets.extend(refined_subj_triplets)
        all_triplets.extend(refined_obj_triplets)

    filtered_triplets = []

    for triplet in all_triplets:
        subj, verb, obj = triplet

        # Remove stopwords
        subj_filtered = ' '.join([w for w in subj.split() if w.lower() not in stopwords]).strip()
        verb_filtered = ' '.join([w for w in verb.split() if w.lower() not in stopwords]).strip()
        obj_filtered = ' '.join([w for w in obj.split() if w.lower() not in stopwords]).strip()

        # Skip remove stopwords if any element becomes empty
        if not subj_filtered:
            subj_filtered = subj
        if not verb_filtered:
            verb_filtered = verb
        if not obj_filtered:
            obj_filtered = obj

        subj_filtered = subj_filtered.replace('_', ' ').strip().lower()
        verb_filtered = verb_filtered.replace('_', ' ').strip().lower()
        obj_filtered = obj_filtered.replace('_', ' ').strip().lower()
        filtered_triplets.append((subj_filtered, verb_filtered, obj_filtered))

    return filtered_triplets



import logging
from src.triplet_extraction.src import setup_logger

os.makedirs(os.path.dirname(log_file_path), exist_ok=True)
logger, console_handler, file_handler = setup_logger(
    name="triplet_extraction",
    level=logging.DEBUG,
    log_to_file=False,
    file_path=log_file_path
)

# Disable console logging (optional)
logger.removeHandler(file_handler)

for sec in sections:
    print(sec["sequence"], sec["content"])
    triplets = triplet_extraction(
                    text=sec["content"],
                    vncorenlp_client=vncorenlp_client,
                    phoNLP_model=phoNLP_model,
                    stopwords=stopwords,
                    logger=logger,
                    max_depth=4,
                )

    print(f"Extracted {len(triplets)} triplets:")
for t in triplets:
    print(t)

NameError: name 'sections' is not defined

In [ ]:
error_csv_file = r"/src/triplet_extraction/logs/no_triplets_dat_dai_log_1.csv"

import pandas as pd
error_df = pd.read_csv(error_csv_file)

unique_document_number = set()
for _, row in error_df.iterrows():
    unique_document_number.add(row['document_number'])

print(len(unique_document_number))
for doc_number in unique_document_number:
    print(doc_number)